# Notebook Training

Questo notebook si occupa del training del modello.

Ѐ stato eseguito su Kaggle, usando una singola GPU P-100.

Il modello di partenza è yolo11s, scelto per la sua versatilità e leggerezza.

### Installazione dei pacchetti necessari

In [ ]:
!pip install -U ultralytics
!pip install gdown
!pip install pyyaml
!pip install roboflow
!pip install --upgrade --force-reinstall sympy



### Creazione sottocartelle
Utilizzate per riprendere il training in caso di mancata persistenza dei file su kaggle

In [ ]:
import os

# Definizione del percorso della cartella principale
base_path = "/kaggle/working/CV-1"

# Definizione delle sottocartelle da creare
subfolders = ["training_run", "training_run/weights"]

# Creazione delle cartelle
for folder in subfolders:
    path = os.path.join(base_path, folder)
    os.makedirs(path, exist_ok=True)
    print(f"Cartella creata: {path}")


### Download dei pesi

Per gestire interruzioni improvvise causate dai limiti temporali imposti dalla piattaforma utilizzata (Kaggle) si è scelto di effettuare dei backup su Google Drive, per poter riprendere l'addestramento dall'ultima epoca calcolata senza perdere informazioni.

In [ ]:
import gdown

file_id = "1pDrXrsL_ayI6DN7o5QO1C5yxkghm5mXr"
destination_path = "/kaggle/working/CV-1/training_run/weights/last.pt"

gdown.download(f"https://drive.google.com/uc?export=download&id={file_id}", destination_path, quiet=False)

print(f"✅ File scaricato in {destination_path}")


# Elaborazione dataset

Si è effettuata una fase di pre-processing e data agumentation usando la piattaforma roboflow. 

I valori utilizzati sono i seguenti:
- **Flip**: Horizontal
- **Rotation**: Between -4° and +4°
- **Saturation**: Between -32% and +32%
- **Brightness**: Between -10% and +10%
- **Exposure**: Between -10% and +10%
- **Blur**: Up to 1.1px
- **Noise**: Up to 0.3% of pixels

Inoltre, il dataset viene scaricato direttamente usando la piattaforma.

In [ ]:

from roboflow import Roboflow
import shutil
import yaml
import zipfile
import os
from pathlib import Path

# Download del dataset da Roboflow

rf = Roboflow(api_key="your_api_key")
project = rf.workspace("computervision-p7hdm").project("cv-lnm6k")
version = project.version(1)
dataset = version.download("yolov11")



dataset_path = dataset.location  

# Definizione delle cartelle di destinazione
train_img_dir = os.path.join(dataset_path, 'images')
train_label_dir = os.path.join(dataset_path, 'labels')
os.makedirs(train_img_dir, exist_ok=True)
os.makedirs(train_label_dir, exist_ok=True)

# Estrazione dei file ZIP se
zip_path = os.path.join(dataset_path, "dataset.zip")
if os.path.exists(zip_path):
    print("Estrazione del dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(dataset_path)



# Creazione del file YAML 
data_yaml = {
    'train': train_img_dir,
    'val': "/kaggle/working/CV-1/valid/images",  
    'names': ["ball"]
    'nc': 1
}

with open(os.path.join(dataset_path, 'data.yaml'), 'w') as f:
    yaml.dump(data_yaml, f)

print("Dataset preparato con successo!")
print(f"Immagini salvate in: {train_img_dir}")
print(f"Etichette salvate in: {train_label_dir}")


## Pulizia GPU
Prima di avviare il training risulta utile terminare tutti i processi superflui, in modo da avere più memoria a disposizione.

In [ ]:
import torch
import gc
import os

def free_gpu():
    print("Liberazione memoria GPU in corso...")
    torch.cuda.empty_cache()  
    gc.collect()  # Forza il garbage collector a rilasciare la memoria

    # Termina i processi zombie
    os.system("kill -9 $(pgrep -f 'python')")  # Forza la chiusura di processi Python
    print("Memoria GPU liberata con successo!")

free_gpu()


Liberazione memoria GPU in corso...


# Training 

Si è scelto di usare come modello di partenza yolo11, in particolare la versione small(yolo11s).
Per ottimizzare la detection di un oggetto piccolo(palla), si è scelto di congelare una delle 3 teste di yolo, ovvero la P5. 
Questa testa corrisponde al livello 22, dettaglio ottenuto controllando il file yaml di yolo11. 
Il compito di questa testa era quello di elaborare oggetti più grandi durante la detection, compito ritenuto inutile per questo caso specifico.
Aver congelato il calcolo dei gradienti di questa testa d'attenzione ha permesso di ottenere una prestazione migliore, velocizzando il calcolo. Le risorse risparmiate sono state usate per aumentare la risoluzione delle immagini di addestramento a 1088 pixel.

In [ ]:
import os
import torch
from pathlib import Path
from dataclasses import dataclass
from typing import Union, List
from ultralytics import YOLO

@dataclass
class TrainingConfig:
    """Configurazione ottimizzata per rilevamento single-class"""
    # Path specifici per Kaggle
    save_path: str = "/kaggle/working/CV-1"
    project_name: str = "training_run"
    data_path: str = "/kaggle/working/CV-1/data.yaml"
    base_model: str = "yolo11s.pt"
    
    epochs: int = 150
    imgsz: int = 1088
    save_period: int = 10
    
    # Data augmentation minima (già preprocessato)
    hsv_h: float = 0.008
    hsv_s: float = 0.1
    hsv_v: float = 0.1
    fliplr: float = 0
    mosaic: float = 0.1
    mixup: float = 0.05
    
    # Ottimizzazione 
    optimizer: str = "AdamW"
    lr0: float = 0.001
    lrf: float = 0.01
    momentum: float = 0.9
    weight_decay: float = 0.01
    warmup_epochs: int = 3
    warmup_momentum: float = 0.8
    warmup_bias_lr: float = 0.1
    
    # Early stopping 
    patience: int = 15                
    close_mosaic: int = 0
    
    # Parametri specifici 
    conf: float = 0.35
    iou: float = 0.5
    max_det: int = 4
    nms: bool = True


class YOLOTrainer:
    def __init__(self, config: TrainingConfig):
        self.config = config
        self.checkpoint_path = Path(config.save_path) / config.project_name / "weights"
        self.device_setting = "cuda"
        self.batch_size = 16
        self.num_workers = 2
        
    
    
    def _find_last_checkpoint(self) -> tuple[str, bool]:
        try:
            if self.checkpoint_path.exists():
                checkpoints = sorted(self.checkpoint_path.glob("last.pt"), 
                                  key=lambda x: x.stat().st_mtime, 
                                  reverse=True)
                if checkpoints:
                    print(f"Riprendendo il training da: {checkpoints[0]}")
                    return str(checkpoints[0]), True
            
            print("Nessun checkpoint trovato, avvio nuovo training.")
            return self.config.base_model, False
            
        except Exception as e:
            print(f"Errore nel caricamento del checkpoint: {e}")
            print("Utilizzo del modello base come fallback.")
            return self.config.base_model, False
    
    def train(self):
        try:
            checkpoint, resume = self._find_last_checkpoint()
            model = YOLO(checkpoint)
            
            results = model.train(
                data=self.config.data_path,
                epochs=self.config.epochs,
                imgsz=self.config.imgsz,
                batch=self.batch_size,
                iou=self.config.iou,
                conf=self.config.conf,
                amp=True,
                augment=True,
                device=self.device_setting,  # Using the selected device (single GPU)
                project=self.config.save_path,
                name=self.config.project_name,
                freeze=[22],
                save_period=self.config.save_period,
                workers=self.num_workers,
                resume=resume,
                hsv_h=self.config.hsv_h,
                hsv_s=self.config.hsv_s,
                hsv_v=self.config.hsv_v,
                fliplr=self.config.fliplr,
                dropout=0.1,
                dfl= 2.0,
                mosaic=self.config.mosaic,
                mixup=self.config.mixup,
                optimizer=self.config.optimizer,
                lr0=self.config.lr0,
                lrf=self.config.lrf,
                max_det=self.config.max_det,
                momentum=self.config.momentum,
                weight_decay=self.config.weight_decay,
                warmup_epochs=self.config.warmup_epochs,
                warmup_momentum=self.config.warmup_momentum,
                warmup_bias_lr=self.config.warmup_bias_lr,
                patience=self.config.patience,
                close_mosaic=self.config.close_mosaic,
                half=True,
                tracker="bytetrack.yaml"
            )
            
            print("Training completato con successo!")
            return results
            
        except Exception as e:
            print(f"Errore durante il training: {e}")
            raise

def main():
    config = TrainingConfig()
    trainer = YOLOTrainer(config)
    trainer.train()

if __name__ == "__main__":
    main()
